In [1]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
import os
import warnings
warnings.filterwarnings("ignore")

In [2]:
RAW_DIR = r"D:\projects\Healthcare\data\raw\\"
SAMPLE_DIR = r"D:\projects\Healthcare\data\raw_sample"

In [3]:
datasets = {
    "diabetes" : {"file":   "diabetes_data.csv",  "target": "diabetes", "size": 20000},
    "cardio"   : {"file":    "cardio_data.csv",       "target": "cardio",   "size": 20000},
    "stroke"   : {"file":    "stroke_data.csv",           "target": "Diagnosis", "size":15000},
}

print("Setup ready. Datasets:", list(datasets.keys()))

Setup ready. Datasets: ['diabetes', 'cardio', 'stroke']


In [4]:

def slice_dataset(name, info):
    
    path = os.path.join(RAW_DIR, info["file"])
    df = pd.read_csv(path)
    target, size = info["target"], info["size"]

    print(f"\n{'='*55}\n  {name.upper()}\n{'='*55}")
    print(f"Original : {df.shape[0]} rows x {df.shape[1]} cols")
    orig = (df[target].value_counts(normalize=True) * 100).round(2)

    if df.shape[0] <= size:
        print(f"--> Already {df.shape[0]} rows (<= {size}). As-is.")
        sliced = df
    else:
        sliced, _ = train_test_split(df, train_size=size,
                                     stratify=df[target], random_state=42)
        print(f"--> Sliced to {size} rows (stratified).")

    new = (sliced[target].value_counts(normalize=True) * 100).round(2)
    print(f"Ratio ('{target}')  original -> sliced:")
    for cls in orig.index:
        print(f"    {str(cls):<14}: {orig[cls]:>6}% -> {new.get(cls, 0):>6}%")
    return sliced


In [5]:
os.makedirs(SAMPLE_DIR, exist_ok=True)  
sliced_datasets = {}
for name, info in datasets.items():
    sliced = slice_dataset(name, info)
    out_path = os.path.join(SAMPLE_DIR, f"{name}_sliced.csv")
    sliced.to_csv(out_path, index=False)
    sliced_datasets[name] = sliced
    print(f"Saved --> {out_path}")


  DIABETES
Original : 100000 rows x 9 cols
--> Sliced to 20000 rows (stratified).
Ratio ('diabetes')  original -> sliced:
    0             :   91.5% ->   91.5%
    1             :    8.5% ->    8.5%
Saved --> D:\projects\Healthcare\data\raw_sample\diabetes_sliced.csv

  CARDIO
Original : 68205 rows x 17 cols
--> Sliced to 20000 rows (stratified).
Ratio ('cardio')  original -> sliced:
    0             :  50.63% ->  50.63%
    1             :  49.37% ->  49.37%
Saved --> D:\projects\Healthcare\data\raw_sample\cardio_sliced.csv

  STROKE
Original : 15000 rows x 22 cols
--> Already 15000 rows (<= 15000). As-is.
Ratio ('Diagnosis')  original -> sliced:
    No Stroke     :  50.21% ->  50.21%
    Stroke        :  49.79% ->  49.79%
Saved --> D:\projects\Healthcare\data\raw_sample\stroke_sliced.csv


In [6]:
SAMPLE_DIR = r"D:\projects\Healthcare\data\raw_sample"
test_file = os.path.join(SAMPLE_DIR, "stroke_sliced.csv")

df = pd.read_csv(test_file)

print("Shape(rows, cols)", df.shape)
print("\nColumns:", df.columns.tolist())

print(df.head())

Shape(rows, cols) (15000, 22)

Columns: ['Patient ID', 'Patient Name', 'Age', 'Gender', 'Hypertension', 'Heart Disease', 'Marital Status', 'Work Type', 'Residence Type', 'Average Glucose Level', 'Body Mass Index (BMI)', 'Smoking Status', 'Alcohol Intake', 'Physical Activity', 'Stroke History', 'Family History of Stroke', 'Dietary Habits', 'Stress Levels', 'Blood Pressure Levels', 'Cholesterol Levels', 'Symptoms', 'Diagnosis']
   Patient ID       Patient Name  Age Gender  Hypertension  Heart Disease  \
0       18153    Mamooty Khurana   56   Male             0              1   
1       62749  Kaira Subramaniam   80   Male             0              0   
2       32145      Dhanush Balan   26   Male             1              1   
3        6154        Ivana Baral   73   Male             0              0   
4       48973  Darshit Jayaraman   51   Male             1              1   

  Marital Status      Work Type Residence Type  Average Glucose Level  ...  \
0        Married  Self-employ

Logic Validation

In [7]:
def validate_dataset(df):
    """Check whether dataset is correct, If problems is there return as list"""
    issues = []

    if df.shape[0] == 0:
        issues.append("Dataset has 0 rows")

    if df.shape[1] == 0:
        issues.append("Dataset has 0 columns")

    if df.shape[1] < 2:
        issues.append("Dataset has less than 2 columns")

    empty_cols = [c for c in df.columns if df[c].isna().all()]
    if empty_cols:
        issues.append(f"Full empty columns:{empty_cols}")

    is_valid = len(issues) == 0
    return is_valid, issues

is_valid, issues = validate_dataset(df)
print("Is it valid:", is_valid)
print("Issues:", issues if issues else "No issues present")



Is it valid: True
Issues: No issues present


In [8]:
def analyze_dataset(df):
    """Generate a complete summary of the dataset."""
    try:
        analysis = {
            "n_rows": df.shape[0],
            "n_cols": df.shape[1],
            "duplicate_rows": int(df.duplicated().sum()),
            "memory_kb": round(df.memory_usage(deep=True).sum() / 1024, 2),
            "columns": [],
        }

        for col in df.columns:
            col_info = {
                "name": col,
                "dtype": str(df[col].dtype),
                "missing": int(df[col].isna().sum()),
                "missing_pct": round(df[col].isna().mean() * 100, 2),
                "unique": int(df[col].nunique()),
            }
            analysis["columns"].append(col_info)

        return analysis

    except Exception as e:
        print(f"Error during analysis: {e}")
        return None


# Test
result = analyze_dataset(df)

print(f"Rows: {result['n_rows']}, Cols: {result['n_cols']}")
print(f"Duplicate rows: {result['duplicate_rows']}")
print(f"Memory: {result['memory_kb']} KB\n")

print(f"{'Column':<22}{'Type':<12}{'Missing':<10}{'Miss%':<8}{'Unique'}")
print("-" * 60)
for c in result["columns"]:
    print(f"{c['name']:<22}{c['dtype']:<12}{c['missing']:<10}{c['missing_pct']:<8}{c['unique']}")

Rows: 15000, Cols: 22
Duplicate rows: 0
Memory: 4667.01 KB

Column                Type        Missing   Miss%   Unique
------------------------------------------------------------
Patient ID            int64       0         0.0     15000
Patient Name          str         0         0.0     13818
Age                   int64       0         0.0     73
Gender                str         0         0.0     2
Hypertension          int64       0         0.0     2
Heart Disease         int64       0         0.0     2
Marital Status        str         0         0.0     3
Work Type             str         0         0.0     4
Residence Type        str         0         0.0     2
Average Glucose Level float64     0         0.0     9215
Body Mass Index (BMI) float64     0         0.0     2490
Smoking Status        str         0         0.0     3
Alcohol Intake        str         0         0.0     4
Physical Activity     str         0         0.0     3
Stroke History        int64       0         0.0  

In [9]:
df.describe()

,Patient ID,Age,Hypertension,Heart Disease,Average Glucose Level,Body Mass Index (BMI),Stroke History,Stress Levels
count,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000
mean,49715.802867,54.035667,0.249000,0.502933,129.445209,27.474302,0.500267,5.022694
std,29000.656642,21.063111,0.432448,0.500008,40.487792,7.230201,0.500017,2.873223
min,1.000000,18.000000,0.000000,0.000000,60.000000,15.010000,0.000000,0.000000
25%,24562.000000,36.000000,0.000000,0.000000,94.517500,21.160000,0.000000,2.540000
50%,49448.000000,54.000000,0.000000,1.000000,128.900000,27.420000,1.000000,5.050000
75%,75112.000000,72.000000,0.000000,1.000000,164.592500,33.720000,1.000000,7.520000
max,99975.000000,90.000000,1.000000,1.000000,200.000000,40.000000,1.000000,10.000000
